# SheafPatternFusion Phase 2.5 - WP2.5.3 discordant-family construction

Grows the seed n3_s03759_d0 / mean(0) into a family over fresh mechanism draws of its frozen structure class. A member is witnessed-discordant iff the engine stays UNDETERMINED while independent samplers exhibit two model-valid completions differing on the target AND a classical witness exists. Gate G2.5d: >= 10 witnessed members greenlights the theory note after G2.5b adjudication.

Code: pip-installed from hugogobato/sheafpatternfusion @ v0.3.0.
Runtime: CPU-only (~2 cores). Expected wall time: < 6 h.

First run: the first cell installs the pinned package and then HALTS with a message. Do Runtime > Restart session once (clears the preloaded numpy/scipy so the ABI-matched binaries load), then Runtime > Run all again; the install cell detects the pins and skips. Later runs on a warm session go straight through.

In [ ]:
import importlib.metadata as md
import subprocess
import sys

WANT = {'numpy': '2.4.3', 'scipy': '1.17.1', 'sheafpatternfusion': '0.3.0'}
TAG = 'v0.3.0'
REPO = 'https://github.com/hugogobato/sheafpatternfusion.git'


def _ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return None


missing = {p: v for p, v in WANT.items() if _ver(p) != v}
if not missing:
    print('environment OK:', WANT)
else:
    print('installing sheafpatternfusion@' + TAG + ' (one-time per session) ...')
    res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                          f'git+{REPO}@{TAG}'])
    if res.returncode != 0:
        raise RuntimeError('pip install failed; see log above')
    print()
    print('=' * 72)
    print('DEPENDENCIES INSTALLED. One manual step left:')
    print('  1) Runtime > Restart session ...   (clears the old numpy/scipy)')
    print('  2) Runtime > Run all               (this cell will skip)')
    print('=' * 72)
    raise SystemExit('restart required before importing numpy/scipy')


In [ ]:
import functools
import json
import multiprocessing as mp
import os
import pathlib
import time

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


In [ ]:
FAMILY_CFG = json.loads(r'''{"description": "WP2.5.3 discordant-family configuration. Structure frozen at the seed n3_s03759_d0 (var_parents {0:[],1:[0],2:[0,1]}; r_parents [[1],[0,2],[0,1,2]]); members are fresh mechanism draws.", "seed_instance_id": "n3_s03759_d0", "target_var": 0, "n_members": 120, "draw_seed_base": 20270901, "sampler_seed": 5150, "engine_jump_starts": 40, "engine_jump_starts_r2": 90, "member_root_starts": 200, "member_max_roots": 16, "member_walk_follows": 4, "member_walk_n_seeds": 10, "member_walk_steps": 60, "phi_tol": 0.0001, "dist_tol": 1e-09, "success_threshold": 10, "outputs": ["results/phase25/discordant_family.jsonl", "docs/discordant_family.md"]}''')


In [ ]:

OUT_DIR = pathlib.Path('/content/results/phase25')
OUT_DIR.mkdir(parents=True, exist_ok=True)


def load_done(path, key_fn):
    done = set()
    if path.exists():
        for line in path.read_text().splitlines():
            try:
                rec = json.loads(line)
                done.add(key_fn(rec))
            except Exception:
                pass
    return done


def pooled_map(worker_fn, items, n_workers=2,
               stall_timeout_s=2400):
    '''Yield worker_fn(item) for all items, 2-process pool with a stall
watchdog: if no future completes within stall_timeout_s, workers are killed
and the remainder runs sequentially. Worker_fn must be picklable (an
importable function or a functools.partial thereof).'''
    from concurrent.futures import ProcessPoolExecutor, FIRST_COMPLETED, wait

    items = list(items)
    if len(items) <= 1 or n_workers <= 1:
        for it in items:
            yield worker_fn(it)
        return
    ctx = mp.get_context('spawn')
    ex = ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx)
    done_count = 0
    try:
        futs = {ex.submit(worker_fn, it): it for it in items}
        pending = set(futs)
        while pending:
            done_set, pending = wait(pending, timeout=stall_timeout_s,
                                     return_when=FIRST_COMPLETED)
            if not done_set:
                raise RuntimeError(
                    f'pool stalled {stall_timeout_s}s with '
                    f'{len(pending)} futures pending')
            for f in done_set:
                done_count += 1
                yield f.result()
        ex.shutdown(wait=False, cancel_futures=True)
    except Exception as e:
        print(f'(pool yielded {done_count}/{len(items)} results, then '
              f'{type(e).__name__}; finishing remainder sequentially)',
              flush=True)
        for proc in (getattr(ex, '_processes', None) or {}).values():
            try:
                proc.kill()
            except Exception:
                pass
        ex.shutdown(wait=False, cancel_futures=True)
        for it in items[done_count:]:
            yield worker_fn(it)


In [ ]:

from sheafpatternfusion.workers import run_family_member

fam_path = OUT_DIR / 'discordant_family.jsonl'
done = load_done(fam_path, lambda r: str(r['member_id']))
pending = [k for k in range(FAMILY_CFG['n_members']) if str(k) not in done]
print(f"{FAMILY_CFG['n_members']} members planned; {len(done)} on file; "
      f"{len(pending)} to go")
# === RUN ===

if pending:
    tp0 = time.time()
    pilot_rec = run_family_member(min(pending), dict(FAMILY_CFG))
    per = time.time() - tp0
    eta_h = per * len(pending) / 2 / 3600
    print(f'self-pilot member {min(pending)}: {per:.0f}s -> projected ~{eta_h:.1f} h '
          f'on 2 workers; continuing', flush=True)

records = []
t0 = time.time()
fout = open(fam_path, 'a')
completed = 0
worker = functools.partial(run_family_member, cfg=dict(FAMILY_CFG))
for rec in pooled_map(worker, pending, n_workers=2):
    fout.write(json.dumps(rec) + '\n')
    records.append(rec)
    completed += 1
    if rec.get('witnessed_discordant'):
        print(f"WITNESSED member {rec['member_id']} "
              f"({rec.get('model_pair_route')})", flush=True)
    if completed % 10 == 0:
        el = time.time() - t0
        print(f'[{completed}/{len(pending)}] {el/60:.1f} min', flush=True)
fout.close()

all_records = [json.loads(l) for l in open(fam_path)]
all_records = [r for r in all_records
               if isinstance(r, dict) and 'member_id' in r]
witnessed = [r for r in all_records if r.get('witnessed_discordant')]
summary = {
    'n_members': len(all_records),
    'n_witnessed_discordant': len(witnessed),
    'success_threshold': FAMILY_CFG['success_threshold'],
    'gate': ('PASS' if len(witnessed) >= FAMILY_CFG['success_threshold']
             else 'COLLAPSE'),
}
(OUT_DIR / 'family_summary.json').write_text(json.dumps(summary, indent=1))
print('FAMILY DONE', summary)


In [ ]:
import glob
output_files = sorted(glob.glob(str(OUT_DIR / '*.jsonl')) + glob.glob(str(OUT_DIR / '*.json')))
for output_file in output_files:
    try:
        from google.colab import files
        files.download(output_file)
        print('Downloaded:', output_file)
    except Exception as e:
        print('(Not on Colab / download skipped):', e)
